# Embed `chunks.jsonl` bằng BGE-m3 trên GPU Colab (T4)

Máy local không có GPU nên bước **embedding** chạy ở đây. Notebook chỉ **tính vector** rồi trả về file
`vectors.npz` (~6 MB); việc nạp vào Qdrant vẫn làm trên máy bạn (`embed_and_index --backend file`).
Không có tài khoản hay mật khẩu nào được đưa lên Colab.

**Cách dùng**
1. `Runtime` → `Change runtime type` → chọn **T4 GPU** → Save.
2. Chạy lần lượt từng cell (`Shift+Enter`).
3. Cell 3 sẽ hỏi file: chọn `data/processed/lao_dong_v5/chunks.jsonl` trên máy bạn.
4. Cell cuối tự tải `vectors.npz` về. Chép nó vào `data/interim/vectors.npz` của dự án.

## 1. Kiểm tra GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "Chưa bật GPU: Runtime > Change runtime type > T4 GPU, rồi chạy lại."
print("GPU:", torch.cuda.get_device_name(0))

## 2. Cài thư viện

In [ ]:
!pip -q install -U sentence-transformers

## 3. Tải `chunks.jsonl` lên

In [ ]:
from google.colab import files

uploaded = files.upload()  # chọn chunks.jsonl
assert uploaded, "Chưa chọn file nào."
CHUNKS_FILE = list(uploaded)[0]
print("Đã nhận:", CHUNKS_FILE)

## 4. Tính vector

- `MODEL` **phải trùng** `EMBEDDING_MODEL` của backend (mặc định `BAAI/bge-m3`). Khác model thì tìm kiếm sẽ sai
  mà không báo lỗi; `embed_and_index` sẽ từ chối file nếu model không khớp.
- Lần đầu cell này tải model (~2,3 GB), mất vài phút. Nếu bị `CUDA out of memory`, giảm `BATCH_SIZE` xuống 4.

In [ ]:
import json

from sentence_transformers import SentenceTransformer

MODEL = "BAAI/bge-m3"
BATCH_SIZE = 8

with open(CHUNKS_FILE, encoding="utf-8") as f:
    chunks = [json.loads(line) for line in f if line.strip()]
ids = [c["chunk_id"] for c in chunks]
texts = [c["text"] for c in chunks]
assert len(set(ids)) == len(ids), "chunk_id bị trùng"
print(len(chunks), "chunks")

model = SentenceTransformer(MODEL, device="cuda")
# normalize_embeddings=True giống hệt embed_local trong embed_and_index.py (Qdrant dùng cosine).
vectors = model.encode(texts, batch_size=BATCH_SIZE, normalize_embeddings=True, show_progress_bar=True)
print("vectors:", vectors.shape)

## 5. Lưu `vectors.npz`

Mỗi vector đi kèm `chunk_id` và mã băm của text. Nhờ vậy `embed_and_index` ghép đúng vector với chunk,
và từ chối nạp nếu `chunks.jsonl` đã đổi sau khi tính.

In [ ]:
import hashlib

import numpy as np


def digest(text):  # phải giống text_digest() trong embed_and_index.py
    return hashlib.sha1(text.encode("utf-8")).hexdigest()[:16]


np.savez_compressed(
    "vectors.npz",
    chunk_ids=np.array(ids),
    text_hash=np.array([digest(t) for t in texts]),
    vectors=vectors.astype(np.float32),
    model=np.array(MODEL),
)

norms = np.linalg.norm(vectors, axis=1)
print("shape:", vectors.shape, "| độ dài vector min/max:", norms.min().round(4), norms.max().round(4), "(phải ≈ 1.0)")
assert np.allclose(norms, 1.0, atol=1e-3)

## 6. Tải kết quả về máy

In [ ]:
from google.colab import files

files.download("vectors.npz")